# 浮点运算（FLOPS）
- 是衡量计算机性能的一个重要指标，尤其是在深度学习和科学计算领域。FLOPS表示每秒浮点运算次数，通常用于评估处理器、GPU和其他计算设备的计算能力。
- 可以采用开源Python库计算，如 thop ,fvcore 和 torchprofile 等。

用随机整数构造一批输入，跑一次前向传播

In [1]:
from transformers import AutoModelForSequenceClassification
import torch
from thop import profile
import time

model = AutoModelForSequenceClassification.from_pretrained(
    "/Users/wjx/Documents/wjx/minimind/minimind-3",
    device_map="mps"
)
# 生成0-1000的随机整数张量，形状为(1, 128)，并将其移动到模型所在的设备上
input_ids = torch.randint(0, 1000, (1, 128)).to(model.device)
# 生成与input_ids形状相同的全1张量作为attention_mask，并将其移动到模型所在的设备上
attention_mask = torch.ones_like(input_ids).to(model.device)

# 计算FLOPS和参数量
flops, params = profile(model,inputs=(input_ids, attention_mask))

start_time = time.time()

# 进行前向传播
with torch.no_grad():
    model(input_ids, attention_mask=attention_mask)

end_time = time.time()

forward_pass_time = end_time - start_time

flops_per_second = flops / forward_pass_time

print(f"FLOPS: {flops_per_second / 1e12:.2f} TFLOPS")   # 大写 S 结尾的 FLOPS = FLOPs per Second，是速率
print(f"FLOPs: {flops / 1e12:.2f} TFLOPs")  # 小写 s 结尾的 FLOPs = Floating Point Operations，是总量。
print(f"参数量: {params / 1e6:.2f} Million")

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.2) or chardet (7.1.0)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
Some weights of Qwen3ForSequenceClassification were not initialized from the model checkpoint at /Users/wjx/Documents/wjx/minimind/minimind-3 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
FLOPS: 0.93 TFLOPS
FLOPs: 0.01 TFLOPs
Number of parameters: 58.98 Million
